# Manila Transit Analytics: Documentation & Replication Guide
This notebook serves as the comprehensive methodology, documentation, and replication guide for the research paper titled **"Manila Transit Analytics: Understanding the Intersection of Time, Weather, and Public Transport Reliability."**

---

## 1. Research Overview & Context
In highly dense metropolitan areas like Metro Manila, public transit reliability is highly sensitive to temporal demand spikes and environmental weather factors. This study evaluates and compares two distinct commuting modalities:
1. **Road-Based Transit**: The **EDSA Carousel** busway corridor.
2. **Rail-Based Transit**: The **MRT-3, LRT-1, and LRT-2** train networks.

### Core Research Questions (RQs)
- **RQ1 (Line Performance)**: Which transit line experiences the longest average actual wait times and absolute delays?
- **RQ2 (Peak Hour Impact)**: How much longer do commuters wait during peak commuter periods (morning/evening rush) compared to off-peak periods?
- **RQ3 (Weather Vulnerability)**: How does precipitation (`Heavy Rain` vs. dry baselines) impact wait times across these systems, and does rail show greater climate resilience than road?
- **RQ4 (Worst Stations)**: Which individual stations represent the worst spatial bottlenecks (highest delay)?

### Mathematical Formulations of Metrics

#### A. Absolute Delay
$$\text{Delay} = \text{Actual Wait Time} - \text{Scheduled Interval}$$

#### B. Delay Ratio (Relative Congestion)
$$\text{Delay Ratio} = \frac{\text{Actual Wait Time}}{\text{Scheduled Interval}}$$

#### C. Percentage Delay Increase (e.g., Weather or Peak)
$$\text{Percentage Increase} = \left( \frac{\text{Mean Wait (Disrupted)} - \text{Mean Wait (Baseline)}}{\text{Mean Wait (Baseline)}} \right) \times 100\%$$



## 2. Data Simulation Methodology
To validate our data cleaning and analysis pipelines, a synthetic dataset was generated by incorporating geographical mapping of station stops and realistic delay logic.

### Math and Distributions of Synthetic Delays
1. **Base Scheduled Interval ($S$)**: Randomly sampled uniformly from $[5, 10]$ minutes.
2. **Time of Day Multiplier ($M_t$)**:
   - **Peak Periods** (`Peak Morning`, `Peak Evening`): $M_t = 2.5$.
   - **Late Night**: $M_t = 0.8$.
   - **Mid-Day**: $M_t = 1.0$.
3. **Weather Delay ($W_d$)**:
   - Under **Heavy Rain**, delays are simulated using a **Gamma Distribution** (capturing the right-skewed nature of transit delays where occasional huge delays occur):
     - **EDSA Carousel (Buses)**: $W_d \sim \text{Gamma}(k=5, \theta=3)$ (Mean delay = 15 minutes).
     - **MRT/LRT (Trains)**: $W_d \sim \text{Gamma}(k=3, \theta=1.5)$ (Mean delay = 4.5 minutes).
   - Under **Clear/Cloudy Weather**: $W_d \sim \text{Gamma}(k=1.5, \theta=1.5)$ (Mean delay = 2.25 minutes).

### Noise Injection Profile
- **Missing Values (Nulls)**: 5% of wait times replaced with `NaN` to simulate sensor logging errors.
- **Categorical Inconsistencies**: 8% of rows have character mutations (swapped case, extra whitespace, missing hyphens) across columns like `Transit_Line`, `Weather_Condition`, etc.
- **Gaussian Outliers**: 20 extreme outliers split between impossible negatives ($N(\mu=-15, \sigma=3)$ mins) and extreme system glitches ($N(\mu=500, \sigma=75)$ mins).
- **Duplicates**: 10 duplicate rows appended.

Below is the generation function implemented in `scripts/data_simulation.py`:


In [ ]:
# Let's view the core configuration for dirty data injection
import numpy as np
import random

CONFIG = {
    'target_rows': 1000,
    'missing_ratio': 0.05,
    'inconsistent_ratio': 0.08,
    'outlier_count': 20,
    'duplicate_count': 10
}
print('Data Simulation configurations initialized.')


## 3. Data Cleaning Pipeline Walkthrough
To resolve the injected noise, we construct a sequential pipeline where **operation order is critical**:

1. **Deduplication First**: Removing duplicates before calculating medians to avoid statistical skew.
2. **Categorical Normalization**: Normalizing string formatting inconsistencies using case-insensitive and spacing/hyphen-insensitive normalization keys.
3. **Median Imputation**: Replacing missing wait times with the group median of the corresponding `Transit_Line` + `Time_of_Day` (median is chosen over mean as it is more robust to outlier values which are still in the dataset at this stage).
4. **Outlier Filtering**: Filtering out impossible negative wait times ($<0.0$) and extreme anomalies ($>120.0$ minutes).
5. **Categorical Validation**: Discarding records with unresolvable categorical values as a final security check.

Let's look at the normalization key function from `scripts/data_cleaning.py`:


In [ ]:
def make_normalization_key(s: str) -> str:
    """Generates a key for case, spacing, and hyphen insensitive comparison."""
    return s.lower().replace('-', '').replace(' ', '')

# Examples
print('mrt-3   ->', make_normalization_key('mrt-3   '))
print('MRT 3   ->', make_normalization_key('MRT 3   '))


## 4. Data Analysis Computations
The analysis script `scripts/data_analysis.py` processes the cleaned dataset and computes statistics for each research question.

### Resolving Casing Anomalies during Aggregation
During analysis, we load the cleaned data and convert all values in `Station_From` to uppercase. This step is critical: because `Station_From` has dozens of entries, minor casing irregularities (e.g. `mAGALLANES` vs `MAGALLANES`) will split a station into separate groups, skewing average delay metrics. Converting to uppercase merges these groups correctly.

Let's load the data, execute the analysis computations, and view the inline results:


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ingest and preprocess cleaned dataset
df = pd.read_csv('../data/2_cleaned/manila_transit_cleaned.csv')
df['Station_From'] = df['Station_From'].astype(str).str.upper()
df['Delay'] = df['Actual_Wait_Time'] - df['Scheduled_Interval']

print(f"Loaded {len(df)} cleaned records for analysis.")


In [ ]:
# RQ1: Overall Line Performance
summary_rq1 = df.groupby('Transit_Line').agg(
    avg_scheduled=('Scheduled_Interval', 'mean'),
    avg_actual=('Actual_Wait_Time', 'mean'),
    avg_delay=('Delay', 'mean')
).reset_index()
print("=== RQ1: OVERALL PERFORMANCE ===")
print(summary_rq1.round(2))


In [ ]:
# RQ2: Peak Hour Impact
peak_map = {
    'Peak Morning': 'Peak',
    'Peak Evening': 'Peak',
    'Mid-Day': 'Off-Peak',
    'Late Night': 'Off-Peak'
}
df['Period_Type'] = df['Time_of_Day'].map(peak_map)
summary_rq2 = df.groupby(['Transit_Line', 'Period_Type'])['Actual_Wait_Time'].mean().unstack()
summary_rq2['Pct_Increase'] = ((summary_rq2['Peak'] - summary_rq2['Off-Peak']) / summary_rq2['Off-Peak']) * 100
print("=== RQ2: PEAK HOUR IMPACT ===")
print(summary_rq2.round(2))


In [ ]:
# RQ3: Weather Vulnerability
summary_rq3 = df.groupby(['Transit_Line', 'Weather_Condition'])['Actual_Wait_Time'].mean().unstack()
summary_rq3['Dry_Avg'] = (summary_rq3['Clear'] + summary_rq3['Cloudy']) / 2
summary_rq3['Pct_Increase_Rain'] = ((summary_rq3['Heavy Rain'] - summary_rq3['Dry_Avg']) / summary_rq3['Dry_Avg']) * 100
print("=== RQ3: WEATHER VULNERABILITY ===")
print(summary_rq3.round(2))


In [ ]:
# RQ4: Worst Stations
station_delays = df.groupby(['Transit_Line', 'Station_From'])['Delay'].mean().reset_index()
top_5_worst = station_delays.sort_values(by='Delay', ascending=False).head(5)
print("=== RQ4: TOP 5 WORST STATIONS ===")
print(top_5_worst.round(2))


### Plotting Visualizations Inline
We generate four charts to visualize our findings:


In [ ]:
# Plot 1: Scheduled vs. Actual
plt.figure(figsize=(8, 4.5))
df_melt = df.melt(id_vars=['Transit_Line'], value_vars=['Scheduled_Interval', 'Actual_Wait_Time'],
                  var_name='Time_Type', value_name='Minutes')
df_melt['Time_Type'] = df_melt['Time_Type'].map({'Scheduled_Interval': 'Scheduled', 'Actual_Wait_Time': 'Actual'})
sns.barplot(data=df_melt, x='Transit_Line', y='Minutes', hue='Time_Type', palette=['#BDC3C7', '#2C3E50'], alpha=0.9)
plt.title("Scheduled vs. Actual Wait Time by Line")
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Peak vs Off-Peak
plt.figure(figsize=(8, 4.5))
sns.barplot(data=df, x='Transit_Line', y='Actual_Wait_Time', hue='Period_Type', hue_order=['Off-Peak', 'Peak'], palette=['#7F8C8D', '#D35400'], alpha=0.9)
plt.title("Average Wait Time: Peak vs. Off-Peak")
plt.tight_layout()
plt.show()


In [ ]:
# Plot 3: Weather Impact
plt.figure(figsize=(8.5, 4.5))
sns.barplot(data=df, x='Transit_Line', y='Actual_Wait_Time', hue='Weather_Condition', hue_order=['Clear', 'Cloudy', 'Heavy Rain'], palette=['#F1C40F', '#95A5A6', '#2980B9'], alpha=0.9)
plt.title("Average Wait Time by Weather Condition")
plt.tight_layout()
plt.show()


In [ ]:
# Plot 4: Top 5 Stations
plt.figure(figsize=(8.5, 4.5))
top_5_worst['Label'] = top_5_worst['Station_From'] + " (" + top_5_worst['Transit_Line'] + ")"
palette_map = {'EDSA Carousel': '#E74C3C', 'MRT-3': '#3498DB', 'LRT-2': '#9B59B6', 'LRT-1': '#2ECC71'}
sns.barplot(data=top_5_worst, y='Label', x='Delay', hue='Transit_Line', palette=palette_map, dodge=False, alpha=0.9)
plt.title("Top 5 Stations with Highest Average Delay")
plt.tight_layout()
plt.show()


## 5. Interactive Dashboard & Custom Server Architecture
To present our findings dynamically without revealing raw data matrices, we developed a local client-side dashboard.

### GUI Core Components (`src/ui/`)
1. **`index.html`**: Structured as a semantic HTML5 template, using Google Fonts (Outfit & Inter) and glassmorphism elements.
2. **`style.css`**: Styling utilizes modern dark mode parameters, glow backdrops (`filter: blur(120px)`), and transitions for active tab states.
3. **`app.js`**: Houses a client-side parser to read the CSV file. It counts records and triggers a count-up animation for `#stat-trips` while maintaining state for plot-swapping tabs.

### Multi-threaded Python Web Server (`main.py`)
To run the GUI dashboard locally, `main.py` starts a background server. The server implements:
- **Custom Request Handler**: A subclass `NoCacheHTTPRequestHandler` inheriting from `http.server.SimpleHTTPRequestHandler` that explicitly overrides `end_headers` to inject `Cache-Control: no-store` headers. This prevents browsers from caching charts when the simulation regenerates raw data.
- **Threading Integration**: Runs the server loop in a background daemon thread (`threading.Thread`) allowing the main thread to open the user's web browser automatically using `webbrowser.open()` without blocking terminal responsiveness.


## 6. End-to-End Execution
You can run the full pipeline (simulation, cleaning, and analysis) directly from the cell below to regenerate all datasets and visuals:


In [ ]:
# Execute entire pipeline programmatically
import subprocess
import sys

print("Starting simulation...")
subprocess.run([sys.executable, '../scripts/data_simulation.py'], check=True)

print("Starting cleaning pipeline...")
subprocess.run([sys.executable, '../scripts/data_cleaning.py'], check=True)

print("Starting exploratory data analysis...")
subprocess.run([sys.executable, '../scripts/data_analysis.py'], check=True)

print("\n=== PIPELINE SUCCESSFULLY EXECUTED AND ALL DATA RE-GENERATED ===")
